In [ ]:
%matplotlib widget
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import matplotlib

# ============================================================
# 1. STYLE CONFIGURATION
# ============================================================
poster_style = {
    "figsize": (5, 5),          
    "dpi": 200,                 
    "font_size": 10,
    "title_size": 12,
    "label_size": 12,
    "legend_size": 8,
    "line_width": 1.0,
    "font_family": "Calibri",
    "grid": True,               
    "colors": ["#0072B2", "#E69F00", "#009E73", "#CC79A7", "#56B4E9", "#D55E00", "#F0E442"], 
    "linestyles": ["-", "--", "-.", ":"]
}

In [ ]:
# ============================================================
# 2. CORE FUNCTIONS
# ============================================================

def PlotSpectra(
        spectra_dict, spectra_names, style=poster_style, offset_step=0.0,
        xmin=None, xmax=None, ymin=None, ymax=None, ylabel="Absorbance",
        custom_labels=None, custom_colors=None, custom_linestyles=None, 
        hide_yticks=False, save_path=None
):
    plt.figure(figsize=style["figsize"], dpi=style["dpi"])
    plt.rcParams.update({
        "font.size": style["font_size"],
        "font.family": style["font_family"]
    })
    
    default_colors = style.get("colors") or ["#000000"]
    default_linestyles = style.get("linestyles") or ["-"]
    
    for i, name in enumerate(spectra_names):
        if name not in spectra_dict:
            print(f"WARNING: {name} not found")
            continue

        df = spectra_dict[name]
        X = df["Wavelength"].values
        Y = df["Abs"].values + (i * offset_step)
        
        label = custom_labels[i] if custom_labels and i < len(custom_labels) else name
        c = custom_colors[i] if custom_colors and i < len(custom_colors) else default_colors[i % len(default_colors)]
        ls = custom_linestyles[i] if custom_linestyles and i < len(custom_linestyles) else default_linestyles[i % len(default_linestyles)]

        plt.plot(X, Y, label=label, linewidth=style["line_width"], color=c, linestyle=ls)

    plt.xlabel("Wavelength (nm)", fontsize=style["label_size"])
    plt.ylabel(ylabel, fontsize=style["label_size"])
    plt.title("UV-Vis-NIR Spectra", fontsize=style["title_size"])

    if xmin is not None or xmax is not None:
        plt.xlim(xmin, xmax)
    if ymin is not None or ymax is not None:
        plt.ylim(ymin, ymax)
        
    plt.grid(style.get("grid", False), alpha=0.3)
    
    if hide_yticks:
        plt.gca().set_yticklabels([])
        
    plt.legend(fontsize=style["legend_size"])
    plt.tight_layout()
    
    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=style["dpi"], bbox_inches="tight")
        print(f"\nFigure saved successfully to:\n{save_path}")
        
    plt.show()

def SubtractBaseline(spectra_input, baseline_df):
    corrected_spectra = {}
    baseline_x = baseline_df["Wavelength"].values
    baseline_y = baseline_df["Abs"].values
    for name, df in spectra_input.items():
        measured_x = df["Wavelength"].values
        measured_y = df["Abs"].values
        baseline_interpolated = np.interp(measured_x, baseline_x, baseline_y)
        corrected = df.copy()
        corrected["Abs"] = measured_y - baseline_interpolated
        corrected_spectra[name] = corrected
    return corrected_spectra

def ApplyCalibrationCorrection(spectra_input, calibration_spectrum, cal_factors):
    corrected_spectra = {}
    calibration_x = calibration_spectrum["Wavelength"].values
    calibration_y = calibration_spectrum["Abs"].values
    for name, df in spectra_input.items():
        if name not in cal_factors: continue
        factor = cal_factors[name]
        measured_x = df["Wavelength"].values
        measured_y = df["Abs"].values
        calibration_interpolated = np.interp(measured_x, calibration_x, calibration_y)
        corrected = df.copy()
        corrected["Abs"] = measured_y - (factor * calibration_interpolated)
        corrected_spectra[name] = corrected
    return corrected_spectra

def NormalizeSpectra(spectra_input, xmin, xmax, mode="M"):
    normalized = {}
    for name, df in spectra_input.items():
        df_norm = df.copy()
        X = df_norm["Wavelength"].values
        Y = df_norm["Abs"].values
        mask = (X >= xmin) & (X <= xmax)
        X_window = X[mask]
        Y_window = Y[mask]
        if len(X_window) == 0: continue
        
        if mode.upper() == "M": factor = np.max(Y_window)
        elif mode.upper() == "I": factor = np.trapezoid(Y_window, X_window)
        elif mode.upper() == "V": factor = np.min(Y_window)
        else: raise ValueError("Mode must be 'M', 'I', or 'V'")
        if factor == 0: continue
            
        df_norm["Abs"] = Y / factor
        normalized[name] = df_norm
    return normalized

def TrimSpectra(spectra_input, xmin=None, xmax=None):
    """
    Trim spectra to a given Wavelength range.
    
    Parameters
    ----------
    spectra_input : dict of pandas DataFrames
    xmin, xmax : float or None
    
    Returns
    -------
    dict of trimmed spectra
    """
    trimmed = {}
    for name, df in spectra_input.items():
        # Start with all True values
        mask = np.ones(len(df), dtype=bool)
        
        if xmin is not None:
            mask &= (df["Wavelength"] >= xmin)
        if xmax is not None:
            mask &= (df["Wavelength"] <= xmax)
            
        df_trimmed = df[mask].copy().reset_index(drop=True)
        
        if df_trimmed.empty:
            print(f"Warning: empty trim range for {name}")
            continue
            
        trimmed[name] = df_trimmed
        
    return trimmed

In [ ]:
# ============================================================
# 3A. LOAD CALIBRATION DATA
# ============================================================
References = r"H:\FUBerlin\Measurements\UVvisNIR\References.csv"
calibration_df = pd.read_csv(
    References, sep=",", skiprows=2, header=None,
    names=["Water_Wavelength", "WaterInD2O", "Nicodenz_Wavelength", "Nicodenz",
           "Empty_Wavelength", "Empty_P2", "WaterFilled_Wavelength", "WaterFilled"]
)

calibration_spectra = {}
for i in range(0, len(calibration_df.columns), 2):
    wavelength_col = calibration_df.columns[i]
    absorbance_col = calibration_df.columns[i + 1]
    spectrum = pd.DataFrame({
        "Wavelength": pd.to_numeric(calibration_df[wavelength_col], errors="coerce"),
        "Abs": pd.to_numeric(calibration_df[absorbance_col], errors="coerce")
    }).dropna().sort_values("Wavelength").reset_index(drop=True)
    calibration_spectra[absorbance_col] = spectrum

water_calibration = calibration_spectra["WaterInD2O"]

# ============================================================
# 3B. LOAD EXPERIMENTAL DATA
# ============================================================
txt_folders = [
    r"H:\FUBerlin\Measurements\UVvisNIR\CristianB\20260810_DGUdial_P2_F005_F006_F007",
    r"H:\FUBerlin\Measurements\UVvisNIR\CristianB\20260817_DGUdial_P2_F005_F006_F007_ProperResolution",
]

txt_spectra = {}
print("Loading Experimental Spectra...")
for txt_folder in txt_folders:
    for file in os.listdir(txt_folder):
        if not file.lower().endswith(".txt"):
            continue
        path = os.path.join(txt_folder, file)
        try:
            df = pd.read_csv(path, sep="\t", skiprows=1, dtype=str)
            df.columns = ["Wavelength", "Abs"]
            df["Wavelength"] = df["Wavelength"].str.strip().str.replace(",", ".", regex=False)
            df["Abs"] = df["Abs"].str.strip().str.replace(",", ".", regex=False)
            df["Wavelength"] = pd.to_numeric(df["Wavelength"], errors="coerce")
            df["Abs"] = pd.to_numeric(df["Abs"], errors="coerce")
            df = df.dropna().sort_values("Wavelength").reset_index(drop=True)
            txt_spectra[file] = df
        except Exception as e:
            print(f"Could not load {file}: {e}")

baseline = txt_spectra["20260817_Baseline_DOCD2O1pc.txt"]

spectra_to_plot = [
    "20260817_F007WA_1BrC18@P2_DGUdial.txt",
    "20260817_F005LP_6FBz@P2_DGUdial.txt",
    "20260817_P2_DGUdial.txt",
    "20260817_F006WA_phDADQ@P2_DGUdial.txt",
]
selected_spectra = {name: txt_spectra[name] for name in spectra_to_plot if name in txt_spectra}
print(f"Loaded {len(selected_spectra)} selected spectra successfully.")

In [ ]:
# ============================================================
# 4. DATA CORRECTION & NORMALIZATION
# ============================================================

# 4A. Baseline Subtraction
baseline_corrected_spectra = SubtractBaseline(selected_spectra, baseline)

# 4B. Water Calibration Correction
calibration_factors = {
    "20260817_F007WA_1BrC18@P2_DGUdial.txt": 0.2,
    "20260817_F005LP_6FBz@P2_DGUdial.txt": 1.5,
    "20260817_P2_DGUdial.txt": 0,
    "20260817_F006WA_phDADQ@P2_DGUdial.txt": 0.3,
}
water_corrected_spectra = ApplyCalibrationCorrection(
    baseline_corrected_spectra, 
    water_calibration, 
    calibration_factors
)

# 4C. Normalization
normalized_spectra = NormalizeSpectra(
    water_corrected_spectra, 
    xmin=1200, 
    xmax=1350, 
    mode="V"
)
print("Data correction and normalization complete.")

In [ ]:
# ============================================================
# 5A. PLOT CALIBRATION SPECTRA
# ============================================================
calibration_to_plot = [
    "WaterInD2O",
    "Nicodenz",
    "Empty_P2",
    "WaterFilled",
]

pathFigures = r"H:\FUBerlin\DataAnalysis\UVvisNIR\NT26_PosterFigures_DGUdialSamples"
cal_output_file = os.path.join(pathFigures, "UVvisNIR_Calibration.svg")

cal_labels = [
    "Water in D2O",
    "Nicodenz",
    "Empty P2",
    "Water Filled P2"
]

my_custom_colors = ["#0072B2", "#D55E00", "#000000", "#009E73"]
my_custom_linestyles = ["-", "-", "-", "-"]

PlotSpectra(
    spectra_dict=calibration_spectra,        
    spectra_names=calibration_to_plot,
    style=poster_style,
    custom_labels=cal_labels,                
    custom_colors=my_custom_colors,          
    custom_linestyles=my_custom_linestyles,  
    offset_step=0.0,
    xmin=220, xmax=1920,
    ymin=-0.1, ymax=0.6,
    ylabel="Absorbance",
    hide_yticks=True,                        
    #save_path=cal_output_file                 
)

In [ ]:
# ============================================================
# 5B. PLOT EXPERIMENTAL SPECTRA
# ============================================================
exp_output_file = os.path.join(pathFigures, "UVvisNIR_DGUdial.svg")

clean_labels = [
    "F007WA",
    "F005LP",
    "P2 Dialyzed",
    "F006WA"
]

PlotSpectra(
    spectra_dict=normalized_spectra,
    spectra_names=spectra_to_plot,
    style=poster_style,
    custom_labels=clean_labels,
    custom_colors=my_custom_colors,          
    custom_linestyles=my_custom_linestyles,  
    offset_step=0.0,
    xmin=220, xmax=1920,
    ymin=-1.5, ymax=11,
    ylabel="Absorbance",
    hide_yticks=True,                        
    save_path=exp_output_file
)

In [ ]:
##### New functions implementation

def BackgroundCorrection(spectra_input, a, b):
    """
    Subtracts a 1/wavelength scattering background.
    Formula: Corrected = Measured - (a * (1 / Wavelength) + b)
    
    Parameters
    ----------
    spectra_input : dict of pandas DataFrames
    a : float or dict (maps filename to float)
        Strength of the 1/wavelength term.
    b : float or dict (maps filename to float)
        Constant offset.
        
    Returns
    -------
    dict of corrected spectra
    """
    corrected_spectra = {}
    
    for name, df in spectra_input.items():
        # Check if a and b are dictionaries (for specific values per file) or single numbers
        current_a = a[name] if isinstance(a, dict) and name in a else (a if not isinstance(a, dict) else 0.0)
        current_b = b[name] if isinstance(b, dict) and name in b else (b if not isinstance(b, dict) else 0.0)
        
        corrected = df.copy()
        X = df["Wavelength"].values
        Y = df["Abs"].values
        
        # Apply the 1/wavelength subtraction
        corrected["Abs"] = Y - ((current_a / X) + current_b)
        
        corrected_spectra[name] = corrected
        
    return corrected_spectra

In [ ]:
trimmedSpectra = TrimSpectra(normalized_spectra, 220, 1850)
noBGSpectra = BackgroundCorrection(trimmedSpectra, 1400, 0)


clean_labels = [
    "F007WA 1",
    "F005LP 2",
    "P2 3",
    "F007WA 4"
]

my_custom_colors = ["#0072B2", "#D55E00", "#000000", "#009E73"]
my_custom_linestyles = ["-", "-", "-", "-"]

PlotSpectra(
    spectra_dict=noBGSpectra,
    spectra_names=spectra_to_plot,
    style=poster_style,
    custom_labels=clean_labels,
    custom_colors=my_custom_colors,          
    custom_linestyles=my_custom_linestyles,  
    offset_step=0.1,
    xmin=230, xmax=1850,
    #ymin=-1.5, ymax=11,
    ylabel="Absorbance",
    hide_yticks=False,                        
    save_path=exp_output_file
)